# Cloud fraction on the Payerne section, one frame per cycle

Standalone version of section 4c of `CLCT_spindown_decomposition.ipynb`: the same figure, the same
transect, but written once per assimilation cycle into a folder so the frames can be animated.

Each frame is a 3x3 panel of mean cloud fraction `CLC` on the west-east section through Payerne:

| | 801 | 802 | 801 - 802 |
|---|---|---|---|
| **`lff(T)`** | first guess | first guess | the two first guesses differenced |
| **`iaf(T)`** | initialized analysis | initialized analysis | the two analyses differenced |
| **`iaf(T) - lff(T)`** | what the cycle did to the cloud | same, other experiment | a difference of differences |

`CLC` exists in `lff` and `iaf` only - neither `inc(T)` nor `laf(T)` carries cloud fraction - so
`iaf(T) - lff(T)` is not the assimilation alone: it also contains the 10 minutes of model integration
inside `iaf(T)`, acting on a diagnosed quantity.

**Colour limits are computed once over every cycle and then frozen**, so a feature that brightens
between two frames is really changing and not just being rescaled. That is why the cycles are all
read before any frame is drawn.

Section 6 adds a second series of frames on the same transect: the **first guess** `lff(T)` itself -
`T`, `QV`, `QC`, `RH` and `CLC`, five columns against three rows - written to its own folder.

This notebook reads only what those two figures need, on the transect columns only, so it does not
need any of the other sections of the main notebook.

## 0. Setup

In [1]:
import os
os.environ["ECCODES_VERSION_CHECK_OFF"] = "1"  # must be set before importing earthkit.data

import pickle
from datetime import datetime, timedelta
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import eccodes            # before earthkit.data: fixes native library resolution
import earthkit.data as ekd

SAVE_FRAMES = True      # write one PNG per cycle into FRAME_DIR
SHOW_FIGURES = False    # also draw each frame inline; leave off for a long run

if SAVE_FRAMES and not SHOW_FIGURES:
    matplotlib.use("Agg")

FRAME_DIR = Path("./figures/clc_section_frames")
FRAME_DIR.mkdir(parents=True, exist_ok=True)
FRAME_DPI = 110         # frame size = figsize * dpi; both fixed, so every PNG has one size

In [2]:
# ============================== what to analyse ==============================
EXP_IDS = ["801", "802"]
BASE_DIR_TMPL = "/store_new/mch/msopr/jdelbeke/ICON_TST/{exp}"
YEAR_SUFFIX = "25"

# On disk for both experiments: 2025-10-04 01 UTC .. 2025-10-10 00 UTC.
DATE_START = datetime(2025, 10, 4, 1)
DATE_END = datetime(2025, 10, 10, 0)

# Every Nth cycle. 1 = every hour, which is what makes a smooth animation.
CYCLE_STRIDE = 1

CYCLE_TIMES = pd.date_range(DATE_START, DATE_END, freq="1h").to_pydatetime().tolist()[::CYCLE_STRIDE]

# Parallel cycle reads. A worker holds one cycle's CLC on the transect only, so this is light.
N_WORKERS = 16

# The first-guess state frames of section 6 go in their own folder, so each series can be
# globbed on its own.
STATE_FRAME_DIR = FRAME_DIR.parent / "lff_state_frames"
STATE_FRAME_DIR.mkdir(parents=True, exist_ok=True)

# MOVERO verification, for the bias panel under each frame. These are FG-det scores: every
# row is a +1 h first guess verified against stations, so this is the first-guess bias in
# calendar time. It carries no lead-time axis and therefore cannot show the spin-up curve
# itself - only how the +1 h bias evolves through the period.
MOVERO_WD = Path("/scratch/mch/jdelbeke/movero/wd/2025_kenda_801_802")
VERIF_SUBSET = "ch-sp"     # station subset: ch (all), ch-am, ch-sp, ch-av
# From the header line of the matching surface-stations_*.lst; used only for labelling.
SUBSET_LABEL = {"ch": "all Swiss stations", "ch-am": "Alps", "ch-sp": "Swiss plateau",
                "ch-av": "Alpine valleys"}
VERIF_PARAM = "CLCT"
VERIF_SCORE = "ME"         # mean error = bias, in octa

# ICON-CH1: 80 layers, so the layer touching the ground is 80 and level numbers count
# downwards. Level 18 is bounded above by the HHL interface at about 10.3 km mean altitude,
# so 18-80 is the shallowest range that covers a 10 km section top everywhere.
LEVELS_OF_INTEREST = list(range(18, 81))   # ground to about 10.3 km

# Section line: the great circle through Payerne and St. Gallen, extended back past Payerne,
# so that Payerne lies exactly on it (a Geneva -> St. Gallen line misses it by 29 km).
PAYERNE = (46.8123, 6.9422)       # radiosonde site; (lat, lon)
SECTION_EAST = (47.4200, 9.3700)  # St. Gallen
SECTION_WEST_KM = 65.0            # how far back past Payerne the section starts
N_TRANSECT = 300
SECTION_ZTOP = 10000.0            # top of the section [m above sea level]

print(f"{len(CYCLE_TIMES)} cycles per experiment, "
      f"{CYCLE_TIMES[0]:%Y-%m-%d %H} UTC to {CYCLE_TIMES[-1]:%Y-%m-%d %H} UTC"
      f"{'' if CYCLE_STRIDE == 1 else f', every {CYCLE_STRIDE}th hour'}")
print(f"frames go to {FRAME_DIR.resolve()}")

144 cycles per experiment, 2025-10-04 01 UTC to 2025-10-10 00 UTC
frames go to /users/jdelbeke/Spin-Up-Down-Project/scripts/figures/clc_section_frames


In [3]:
# ============================ plotting conventions ============================
# Kept identical to the main notebook so the frames match the figures there.
EXP_COLOUR = {"801": "#0072B2", "802": "#D55E00"}    # as in the main notebook

# Terrain. Light, so that dark low cloud sitting on it stays legible; the outline keeps the
# ridge line readable now that the fill no longer carries it.
TERRAIN_FILL = "0.82"
TERRAIN_EDGE = "0.35"

CLC_ROWS = [("lff", "lff(T) = first guess"),
            ("iaf", "iaf(T) = initialized ana"),
            ("diff", "iaf(T) - lff(T)")]
CLC_STAGES = ["lff", "iaf"]                     # the only two files that carry CLC
EXP_DIFF = f"{EXP_IDS[0]} - {EXP_IDS[1]}"       # third column: the experiments differenced
CLC_COLS = EXP_IDS + [EXP_DIFF]


# Section 6: the first-guess state. QV and QC are what the cloud scheme is computed from, RH
# is derived from T, QV and P, and CLC is the cloud fraction it diagnoses - so the columns run
# from the inputs to the output.
STATE_ROWS = ["T", "QV", "QC", "RH", "CLC"]
STATE_READ = ["T", "QV", "QC", "P"]              # P is only a stepping stone to RH
VAR_UNITS = {"T": "K", "QV": "kg/kg", "QC": "kg/kg", "RH": "%", "CLC": "%"}
# One sequential map per variable, so a row is recognisable at a glance. The difference
# column keeps PuOr_r throughout, as in the CLC figure.
VAR_CMAP = {"T": "inferno", "QV": "YlGnBu", "QC": "Blues", "RH": "PuBu", "CLC": "Blues"}

# The increment beside the state it was added to. inc(T) carries no cloud fraction and RH is
# not linear in T and QV, so only the three variables the increment actually holds appear.
INC_VARS = ["T", "QV", "QC"]
STATE_COLS = [("lff", v) for v in STATE_ROWS] + [("inc", v) for v in INC_VARS]
SRC_LABEL = {"lff": "lff(T)", "inc": "inc(T)"}


def sym_limit(arrays, pct=99.0):
    """One symmetric colour limit shared across panels, so magnitudes stay comparable."""
    vals = np.concatenate([np.abs(a[np.isfinite(a)]).ravel() for a in arrays])
    v = np.nanpercentile(vals, pct) if vals.size else 1.0
    return v if v > 0 else 1.0

## 1. Reading the files

In [4]:
def file_path(exp_id, kind, time):
    """kind: "lff"/"iff" (first-guess archive) or "inc"/"iaf"/"laf" (analysis archive)."""
    base = BASE_DIR_TMPL.format(exp=exp_id)
    sub = ("FG" if kind in ("lff", "iff") else "ANA") + YEAR_SUFFIX
    return os.path.join(base, sub, "det", f"{kind}{time:%Y%m%d%H}")


# Same cache directory as the main notebook, so an index built there is reused here.
CACHE_DIR = Path(f"/scratch/mch/{os.environ['USER']}/spindown_grib_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def build_index(path):
    """shortName -> [(level, byte offset, byte length), ...], sorted by level.

    A headers-only pass: ecCodes is asked for each message's metadata but never for its
    values, so all this records is where every message sits in the file.
    """
    index = {}
    with open(path, "rb") as f:
        while True:
            h = eccodes.codes_grib_new_from_file(f, headers_only=True)
            if h is None:
                break
            index.setdefault(eccodes.codes_get(h, "shortName"), []).append(
                (eccodes.codes_get_long(h, "level"),
                 eccodes.codes_get_message_offset(h),
                 eccodes.codes_get_long(h, "totalLength")))
            eccodes.codes_release(h)
    for entries in index.values():
        entries.sort()          # by level, so a profile comes back top-down consistently
    return index


def load_index(path):
    """build_index, cached on disk. Key is (name, size, mtime): a regenerated file
    invalidates its own entry instead of returning a stale index."""
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    stat = os.stat(path)
    cached = CACHE_DIR / f"{Path(path).name}_{stat.st_size}_{stat.st_mtime_ns}.pkl"
    if cached.exists():
        with open(cached, "rb") as fh:
            return pickle.load(fh)
    index = build_index(path)
    tmp = cached.with_suffix(f".tmp{os.getpid()}")   # unique + atomic: workers run in parallel
    with open(tmp, "wb") as fh:
        pickle.dump(index, fh, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, cached)
    return index


_grid = None


def grid_lonlat(path):
    """Cell longitudes and latitudes, via earthkit once and cached thereafter.

    ecCodes will not expand the coordinates of this unstructured grid, so one field has to
    go through earthkit. Same .npz as the main notebook.
    """
    global _grid
    if _grid is not None:
        return _grid
    npz = CACHE_DIR / "icon_grid_lonlat.npz"
    if npz.exists():
        z = np.load(npz)
        _grid = (z["lon"], z["lat"])
    else:
        xa = ekd.from_source("file", path).to_fieldlist()[0].to_xarray()
        _grid = (xa["longitude"].values, xa["latitude"].values)
        np.savez(npz, lon=_grid[0], lat=_grid[1])
    return _grid


class Grib:
    """One GRIB file, indexed by variable name, decoding only the messages asked for."""

    def __init__(self, path):
        self.path = path
        self.index = load_index(path)

    def _read(self, entries):
        values = []
        with open(self.path, "rb") as f:
            for _, offset, length in entries:
                f.seek(offset)
                h = eccodes.codes_new_from_message(f.read(length))
                values.append(eccodes.codes_get_values(h))
                eccodes.codes_release(h)
        return np.array([e[0] for e in entries]), np.stack(values)

    def profile(self, short_name, levels=None):
        """Multi-level field -> (levels, array of shape (n_levels, n_cells))."""
        entries = self.index[short_name]
        if levels is not None:
            want = {int(lev) for lev in levels}
            entries = [e for e in entries if e[0] in want]
        return self._read(entries)

    def field(self, short_name):
        """Single-level field -> values."""
        return self._read(self.index[short_name][:1])[1][0]

    @property
    def lonlat(self):
        return grid_lonlat(self.path)

## 2. The section line and its geometry

In [5]:
def initial_bearing(p0, p1):
    """Initial great-circle bearing from p0 to p1 [degrees from north]."""
    la1, lo1, la2, lo2 = map(np.deg2rad, [p0[0], p0[1], p1[0], p1[1]])
    dlo = lo2 - lo1
    return np.rad2deg(np.arctan2(np.sin(dlo) * np.cos(la2),
                                 np.cos(la1) * np.sin(la2) -
                                 np.sin(la1) * np.cos(la2) * np.cos(dlo)))


def destination(p, bearing_deg, dist_km, R=6371.0):
    """Point reached from p by travelling dist_km along a great circle on the given bearing."""
    la1, lo1 = np.deg2rad(p[0]), np.deg2rad(p[1])
    br, d = np.deg2rad(bearing_deg), dist_km / R
    la2 = np.arcsin(np.sin(la1) * np.cos(d) + np.cos(la1) * np.sin(d) * np.cos(br))
    lo2 = lo1 + np.arctan2(np.sin(br) * np.sin(d) * np.cos(la1),
                           np.cos(d) - np.sin(la1) * np.sin(la2))
    return (float(np.rad2deg(la2)), float(np.rad2deg(lo2)))


def transect_points(p0, p1, n):
    """n points along the great circle from p0 to p1; returns lat, lon, distance [km]."""
    la0, lo0, la1, lo1 = map(np.deg2rad, [p0[0], p0[1], p1[0], p1[1]])
    d = np.arccos(np.clip(np.sin(la0) * np.sin(la1) +
                          np.cos(la0) * np.cos(la1) * np.cos(lo1 - lo0), -1, 1))
    f = np.linspace(0, 1, n)
    A, B = np.sin((1 - f) * d) / np.sin(d), np.sin(f * d) / np.sin(d)
    x = A * np.cos(la0) * np.cos(lo0) + B * np.cos(la1) * np.cos(lo1)
    y = A * np.cos(la0) * np.sin(lo0) + B * np.cos(la1) * np.sin(lo1)
    z = A * np.sin(la0) + B * np.sin(la1)
    return (np.rad2deg(np.arctan2(z, np.sqrt(x ** 2 + y ** 2))),
            np.rad2deg(np.arctan2(y, x)), d * 6371.0 * f)


def _unit_xyz(lat, lon):
    la, lo = np.deg2rad(lat), np.deg2rad(lon)
    return np.column_stack([np.cos(la) * np.cos(lo), np.cos(la) * np.sin(lo), np.sin(la)])


def nearest_cells(lon, lat, lon_t, lat_t):
    """Nearest model cell to each requested point, on the sphere."""
    return cKDTree(_unit_xyz(lat, lon)).query(_unit_xyz(lat_t, lon_t))[1]


# Walk back from Payerne along the reverse Payerne -> St. Gallen bearing.
SECTION_WEST = destination(PAYERNE, initial_bearing(PAYERNE, SECTION_EAST) + 180.0,
                           SECTION_WEST_KM)
TRANSECT = (SECTION_WEST, SECTION_EAST)
lat_t, lon_t, dist_t = transect_points(*TRANSECT, N_TRANSECT)

# Static geometry: which cells the section passes through, and their layer heights. The grid
# and the vertical coordinate are the same in both experiments and every cycle, so this is
# read once from the first cycle.
_lff = Grib(file_path(EXP_IDS[0], "lff", CYCLE_TIMES[0]))
_iaf = Grib(file_path(EXP_IDS[0], "iaf", CYCLE_TIMES[0]))
_lon, _lat = _lff.lonlat
t_idx = nearest_cells(_lon, _lat, lon_t, lat_t)

_lev = np.array(sorted(LEVELS_OF_INTEREST))
# Layer L is bounded by HHL interfaces L and L+1: one interface more than levels.
_, _hhl = _iaf.profile("HHL", range(int(_lev.min()), int(_lev.max()) + 2))
z_sec = 0.5 * (_hhl[:-1] + _hhl[1:])[:, t_idx]     # layer-centre altitude [m above sea level]
terrain = _lff.field("HSURF")[t_idx]
dist2d = np.broadcast_to(dist_t, z_sec.shape)

# where Payerne sits along the section
d_pay = np.hypot(np.deg2rad(lat_t - PAYERNE[0]) * 6371.0,
                 np.deg2rad(lon_t - PAYERNE[1]) * 6371.0 * np.cos(np.deg2rad(PAYERNE[0])))
i_pay = int(np.argmin(d_pay))
print(f"section length {dist_t[-1]:.0f} km; Payerne at {dist_t[i_pay]:.0f} km along it, "
      f"{d_pay[i_pay]:.1f} km off the line")
print(f"{len(_lev)} levels kept, {z_sec.min():.0f}-{z_sec.max():.0f} m above sea level")

section length 261 km; Payerne at 65 km along it, 0.4 km off the line
63 levels kept, 413-10181 m above sea level


## 3. Read CLC for every cycle

One worker per (experiment, cycle): two files, `len(LEVELS_OF_INTEREST)` messages each, kept on the
transect columns only. Every cycle is held in memory because the colour limits have to be known
before the first frame is drawn - at 41 levels x 300 columns a cycle is about 50 kB, so the whole
period is a few tens of MB.

In [6]:
def cycle_clc(exp_id, t, cells=None, levels=None):
    """CLC on the transect columns for one cycle: {stage: (n_levels, n_transect)} in %."""
    cells = t_idx if cells is None else cells
    levels = LEVELS_OF_INTEREST if levels is None else levels
    out = {}
    for stage in CLC_STAGES:
        grib = Grib(file_path(exp_id, stage, t))
        out[stage] = grib.profile("CLC", levels)[1][:, cells].astype(np.float32)
    return out


def _clc_job(job):
    exp_id, t = job
    try:
        return exp_id, t, cycle_clc(exp_id, t)
    except Exception as e:
        print(f"  skipped {exp_id} {t:%Y-%m-%d %H}: {type(e).__name__}: {e}", flush=True)
        return exp_id, t, None


jobs = [(e, t) for e in EXP_IDS for t in CYCLE_TIMES]
print(f"reading {len(jobs)} cycle-experiment combinations on {N_WORKERS} workers, "
      f"2 files each, {len(LEVELS_OF_INTEREST)} messages per file")

clc = {}                                    # (exp, time, stage) -> (n_levels, n_transect)
with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = [ex.submit(_clc_job, j) for j in jobs]
    for done, fut in enumerate(as_completed(futures), 1):
        exp_id, t, res = fut.result()
        if res is None:
            continue
        for stage, arr in res.items():
            clc[(exp_id, t, stage)] = arr
        if done % 24 == 0:
            print(f"  {done}/{len(jobs)}", flush=True)

# Only cycles complete in both experiments can be drawn, since every frame shows both.
FRAME_TIMES = [t for t in CYCLE_TIMES
               if all((e, t, s) in clc for e in EXP_IDS for s in CLC_STAGES)]
print(f"\n{len(FRAME_TIMES)} of {len(CYCLE_TIMES)} cycles complete in both experiments")
print(f"held in memory: {sum(a.nbytes for a in clc.values()) / 1e6:.0f} MB")

reading 288 cycle-experiment combinations on 16 workers, 2 files each, 63 messages per file
  24/288
  48/288
  72/288
  96/288
  120/288
  144/288
  168/288
  192/288
  216/288
  240/288
  264/288
  288/288

144 of 144 cycles complete in both experiments
held in memory: 44 MB


## 4. The verification bias timeseries

The panel under every frame. `time_scores01_CLCT.dat` is a MOVERO ATAB file: a variable-length
header, then 144 hourly rows of scores for the +1 h first guess verified against Swiss stations.
`ME` is the bias in octa.

Two things this data cannot do, worth keeping in view while watching the frames:

- **No lead-time axis.** Every row is `lt_hh = 1`, so the spin-up/spin-down signature - error
  evolving over forecast hours 0-6 - is not in these files. What moves along the panel is the +1 h
  first-guess bias in calendar time. Forecast-suite verifications would be needed for the other.
- **Different geometry from the section.** The bias is over station points across all of
  Switzerland; the frames above are one west-east line. They answer related but not identical
  questions, so a mismatch between them is not a contradiction.

In [7]:
def read_movero_time_scores(exp_dir, param=VERIF_PARAM):
    """One MOVERO ATAB time_scores file as a DataFrame indexed by valid time.

    The header block is variable length, so the column names are taken from the row that
    starts with YYYY rather than from a fixed line number. The file's own missing-value
    sentinel (-0.9999e9) becomes NaN.
    """
    path = Path(exp_dir) / f"time_scores01_{param}.dat"
    with open(path) as fh:
        lines = fh.readlines()
    i_hdr = next(i for i, line in enumerate(lines) if line.split()[:1] == ["YYYY"])
    names = lines[i_hdr].split()
    df = pd.read_csv(path, sep=r"\s+", skiprows=i_hdr + 1, names=names, header=None)
    df = df.mask(df < -1e8)                     # the missing-value sentinel
    df["time"] = pd.to_datetime(dict(year=df["YYYY"], month=df["MM"], day=df["DD"],
                                     hour=df["hh"], minute=df["mm"]))
    return df.set_index("time").sort_index()


bias_df = {e: read_movero_time_scores(MOVERO_WD / f"{e}-FG-det_{VERIF_SUBSET}")
           for e in EXP_IDS}
bias = {e: df[VERIF_SCORE] for e, df in bias_df.items()}

for e, df in bias_df.items():
    lt = sorted(df["lt_hh"].dropna().unique())
    print(f"{e}: {len(df)} rows, {df.index[0]:%d %b %H} - {df.index[-1]:%d %b %H} UTC, "
          f"lead time {lt} h, {int(df['N'].max())} stations at most, "
          f"{VERIF_SCORE} mean {bias[e].mean():+.3f} octa "
          f"({bias[e].notna().sum()} valid hours)")

# Fixed across every frame: the marker has to be comparable from one PNG to the next.
BIAS_TSPAN = (min(b.index[0] for b in bias.values()),
              max(b.index[-1] for b in bias.values()))
_all = np.concatenate([b.dropna().values for b in bias.values()])
_pad = 0.12 * (np.nanmax(_all) - np.nanmin(_all))
BIAS_YLIM = (np.nanmin(_all) - _pad, np.nanmax(_all) + _pad)

# The scores are the first guess verified at its valid time, which is the cycle time of the
# frames above, so the two share one clock. The ranges differ by an hour at each end.
_matched = [t for t in CYCLE_TIMES if any(pd.Timestamp(t) in b.index for b in bias.values())]
print(f"\n{len(_matched)} of {len(CYCLE_TIMES)} cycles have a score row; "
      f"frames outside {BIAS_TSPAN[0]:%d %b %H} - {BIAS_TSPAN[1]:%d %b %H} UTC "
      "get the line but no marker")

801: 144 rows, 04 Oct 02 - 10 Oct 01 UTC, lead time [np.int64(1)] h, 14 stations at most, ME mean +0.849 octa (143 valid hours)
802: 144 rows, 04 Oct 02 - 10 Oct 01 UTC, lead time [np.int64(1)] h, 14 stations at most, ME mean +0.629 octa (143 valid hours)

143 of 144 cycles have a score row; frames outside 04 Oct 02 - 10 Oct 01 UTC get the line but no marker


In [8]:
def draw_bias_panel(ax, t=None, legend=True):
    """The bias timeseries, optionally with the marker for cycle `t`.

    Axis limits are the frozen ones, not data-driven, so the marker sits at the same place
    for the same time in every frame.
    """
    for e in EXP_IDS:
        ax.plot(bias[e].index, bias[e].values, "-", color=EXP_COLOUR[e], lw=1.3,
                label=f"{e}  {VERIF_PARAM} {VERIF_SCORE}")
    ax.axhline(0, color="k", lw=0.9)
    ax.set_xlim(*BIAS_TSPAN)
    ax.set_ylim(*BIAS_YLIM)
    ax.grid(alpha=0.3)
    ax.set_ylabel(f"{VERIF_PARAM} bias [octa]")
    if legend:
        ax.legend(fontsize=8, ncol=len(EXP_IDS), loc="upper right")

    if t is None:
        return
    ts = pd.Timestamp(t)
    ax.axvline(ts, color="k", lw=1.4, ls=":", zorder=4)
    shown = []
    for e in EXP_IDS:
        val = bias[e].get(ts, np.nan)
        if np.isfinite(val):
            ax.plot([ts], [val], marker="*", ms=26, color=EXP_COLOUR[e],
                    mec="k", mew=0.9, zorder=6, clip_on=False)
            shown.append(f"{e} {val:+.2f}")
    ax.set_title(f"first-guess {VERIF_PARAM} bias, "
                 f"{SUBSET_LABEL.get(VERIF_SUBSET, VERIF_SUBSET)}; "
                 + ("now: " + ",  ".join(shown) + " octa" if shown
                    else "no verification at this hour"),
                 fontsize=10, loc="left")


# The timeseries on its own, for reference; every frame carries this same panel.
fig, ax = plt.subplots(figsize=(21, 4))
draw_bias_panel(ax)
ax.set_title(f"First-guess {VERIF_PARAM} bias, "
             f"{SUBSET_LABEL.get(VERIF_SUBSET, VERIF_SUBSET)} ({VERIF_SUBSET}), "
             f"{VERIF_SCORE} in octa", fontsize=11, loc="left")
ax.set_xlabel("valid time [UTC]")
     
plt.show() if SHOW_FIGURES else plt.close(fig)

## 5. One figure per cycle

`clc_field` and the 3x3 panel layout are the same as section 4c of the main notebook, with the bias
timeseries added underneath and the marker moved to the cycle being drawn. Two differences from the
main notebook, both for the animation:

- **Colour limits, and the bias panel's axes, are global.** They are taken over every frame at once,
  so a panel getting darker from one frame to the next means more cloud, not a rescaled colour bar,
  and the marker moves against a fixed background.
- **The figure size is fixed and `bbox_inches="tight"` is not used**, so every PNG comes out at the
  same pixel size.

In [9]:
def clc_field(row, exp_id, t):
    """One row of the figure for one experiment at one cycle."""
    if row == "diff":
        return clc[(exp_id, t, "iaf")] - clc[(exp_id, t, "lff")]
    return clc[(exp_id, t, row)]


# Colour limits, over every frame at once and then frozen.
# States: 0-100 %, the full range of a cloud fraction, so the scale cannot drift at all.
# Differences: the 99th percentile of the magnitude over all frames, one limit per row. The
# experiment-difference column gets its own limit and its own diverging map, being a difference
# of differences in the bottom row and an order of magnitude smaller than the states above.
row_kw, diff_kw = {}, {}
for row, _ in CLC_ROWS:
    if row == "diff":
        v = sym_limit([clc_field(row, e, t) for e in EXP_IDS for t in FRAME_TIMES])
        row_kw[row] = dict(cmap="RdBu_r", vmin=-v, vmax=v)
    else:
        row_kw[row] = dict(cmap="Blues", vmin=0, vmax=100)
    vd = sym_limit([clc_field(row, EXP_IDS[0], t) - clc_field(row, EXP_IDS[1], t)
                    for t in FRAME_TIMES])
    diff_kw[row] = dict(cmap="PuOr_r", vmin=-vd, vmax=vd)

for row, label in CLC_ROWS:
    print(f"{label:26s} states/diff +-{row_kw[row]['vmax']:5.1f} %   "
          f"{EXP_DIFF} +-{diff_kw[row]['vmax']:5.1f} %")


def clc_section_figure(t):
    """The 3x3 section figure for one cycle, with the bias panel underneath."""
    fig = plt.figure(figsize=(21, 15))
    # One extra row, shorter than the sections, spanning all three columns.
    gs = fig.add_gridspec(len(CLC_ROWS) + 1, len(CLC_COLS),
                          height_ratios=[1] * len(CLC_ROWS) + [0.45])
    # sharex/sharey by hand: add_subplot has no sharey= across a whole grid, and the bias
    # panel must be left out of the sharing - its axes are time and octa, not km and metres.
    axes = np.empty((len(CLC_ROWS), len(CLC_COLS)), dtype=object)
    for r in range(len(CLC_ROWS)):
        for c in range(len(CLC_COLS)):
            ref = axes[0, 0] if (r or c) else None
            axes[r, c] = fig.add_subplot(gs[r, c], sharex=ref, sharey=ref)
            axes[r, c].label_outer()        # inner tick labels off, as plt.subplots does
    ax_bias = fig.add_subplot(gs[len(CLC_ROWS), :])

    for r, (row, row_label) in enumerate(CLC_ROWS):
        for c, col in enumerate(CLC_COLS):
            ax = axes[r, c]
            if col == EXP_DIFF:
                field = clc_field(row, EXP_IDS[0], t) - clc_field(row, EXP_IDS[1], t)
                kw, cb_label = diff_kw[row], f"{EXP_DIFF} of {row_label}"
            else:
                field, kw, cb_label = clc_field(row, col, t), row_kw[row], row_label
            pc = ax.pcolormesh(dist2d, z_sec, field, shading="auto", **kw)
            ax.fill_between(dist_t, 0, terrain, color=TERRAIN_FILL, zorder=5)
            ax.plot(dist_t, terrain, color=TERRAIN_EDGE, lw=0.8, zorder=5)
            ax.axvline(dist_t[i_pay], color="k", ls=":", lw=1.2, zorder=6)
            ax.text(dist_t[i_pay], SECTION_ZTOP * 0.95, " Payerne", fontsize=8,
                    va="top", ha="left", zorder=7)
            cb = plt.colorbar(pc, ax=ax, shrink=0.85, pad=0.02)
            cb.set_label(f"CLC, {cb_label}  [%]", fontsize=8)
            ax.set_title(f"CLC {row_label}; {col}", fontsize=10)
            ax.set_ylim(0, SECTION_ZTOP)
            if r == len(CLC_ROWS) - 1:
                ax.set_xlabel(f"distance along section [km]\n"
                              f"{TRANSECT[0][0]:.2f}N {TRANSECT[0][1]:.2f}E (Lake Geneva)"
                              f"  ->  {TRANSECT[1][0]:.2f}N {TRANSECT[1][1]:.2f}E (St. Gallen)")
            if c == 0:
                ax.set_ylabel("altitude [m above sea level]")

    draw_bias_panel(ax_bias, t)
    ax_bias.set_xlabel("valid time [UTC]   |   star = the cycle drawn above")

    plt.suptitle("Cloud fraction CLC on the west-east section through Payerne\n"
                 f"valid {t:%a %d %b %Y  %H} UTC   |   bottom row: red = analysis cloudier "
                 f"than first guess; third column: orange = {EXP_IDS[0]} cloudier than "
                 f"{EXP_IDS[1]}; grey = terrain")
    plt.tight_layout()
    return fig

lff(T) = first guess       states/diff +-100.0 %   801 - 802 +- 75.6 %
iaf(T) = initialized ana   states/diff +-100.0 %   801 - 802 +- 78.5 %
iaf(T) - lff(T)            states/diff +- 38.2 %   801 - 802 +- 43.8 %


In [10]:
# Write the frames. Names are zero-padded and sortable, so a glob feeds them to any encoder
# in chronological order.
for n, t in enumerate(FRAME_TIMES, 1):
    fig = clc_section_figure(t)
    if SAVE_FRAMES:
        # no bbox_inches="tight": that trims to the drawn content and so varies by a pixel
        # or two between frames, which encoders reject
        fig.savefig(FRAME_DIR / f"clc_section_{t:%Y%m%d%H}.png", dpi=FRAME_DPI)
    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig)
    if n % 12 == 0 or n == len(FRAME_TIMES):
        print(f"  {n}/{len(FRAME_TIMES)} frames", flush=True)

frames = sorted(FRAME_DIR.glob("clc_section_*.png"))
sizes = {tuple(plt.imread(f).shape[:2]) for f in frames[:5]}
print(f"\n{len(frames)} PNGs in {FRAME_DIR.resolve()}")
print(f"frame size {sizes} (one entry = every frame the same size, as an encoder needs)")

  12/144 frames
  24/144 frames
  36/144 frames
  48/144 frames
  60/144 frames
  72/144 frames
  84/144 frames
  96/144 frames
  108/144 frames
  120/144 frames
  132/144 frames
  144/144 frames

144 PNGs in /users/jdelbeke/Spin-Up-Down-Project/scripts/figures/clc_section_frames
frame size {(1650, 2310)} (one entry = every frame the same size, as an encoder needs)


## 6. The first-guess state on the same section

The second series of frames: the state the assimilation starts from, `lff(T)`, on the same transect.
Five columns - `T`, `QV`, `QC`, `RH`, `CLC` - against three rows, 801, 802 and the two differenced,
with the same bias panel and marker underneath.

To the right of those, three more columns hold the **assimilation increment** `inc(T)` for `T`, `QV`
and `QC` at the same cycle: the observation-driven correction about to be added to the state on the
left. So the figure reads left to right as the state, then what the assimilation does to it. `inc(T)`
carries no cloud fraction and no cloud ice, so there are three increment columns and not five.

`RH` is not in either file - it is formed per column from the first guess `T`, `QV` and `P` with the
same Tetens saturation formula the main notebook uses, so it is consistent with the profiles there.
There is no increment RH column: RH is not linear in T and QV, so a difference of increments would
not be one.

This pairs directly with the bias panel: both describe `lff(T)`, the +1 h first guess, so section and
score are the same model state at the same moment - on different geometry, one line against a set of
stations.

Its `CLC` column repeats the top-left panel of the section-5 figure by design; the point here is to
see it beside the `T`, `QV`, `QC` and `RH` it was diagnosed from.

This is the expensive section of the notebook: four variables on every level of the first-guess file
plus three of the increment, against the two CLC reads of section 3. The two reads are separate
cells, so re-running one does not repeat the other.

In [11]:
def saturation_qv(T_k, p_pa):
    """Saturation specific humidity from temperature [K] and pressure [Pa] (Tetens, over water)."""
    T_c = T_k - 273.15
    e_s = 611.2 * np.exp(17.62 * T_c / (243.12 + T_c))
    return 0.622 * e_s / np.clip(p_pa - 0.378 * e_s, 1.0, None)


def cycle_lff_state(exp_id, t, cells=None, levels=None):
    """The first-guess state on the transect columns: T, QV, QC and derived RH.

    CLC is not read here - section 3 already has it for lff(T) - and P is dropped once RH is
    formed, being of no interest on its own.
    """
    cells = t_idx if cells is None else cells
    levels = LEVELS_OF_INTEREST if levels is None else levels
    lff = Grib(file_path(exp_id, "lff", t))
    out = {v: lff.profile(v, levels)[1][:, cells] for v in STATE_READ}
    out["RH"] = 100.0 * out["QV"] / saturation_qv(out["T"], out["P"])
    del out["P"]
    return {v: a.astype(np.float32) for v, a in out.items()}


def _state_job(job):
    exp_id, t = job
    try:
        return exp_id, t, cycle_lff_state(exp_id, t)
    except Exception as e:
        print(f"  skipped {exp_id} {t:%Y-%m-%d %H}: {type(e).__name__}: {e}", flush=True)
        return exp_id, t, None


print(f"reading {len(jobs)} cycle-experiment combinations on {N_WORKERS} workers, "
      f"1 file each, {len(STATE_READ)} x {len(LEVELS_OF_INTEREST)} messages")

state = {}                                  # (exp, time, var) -> (n_levels, n_transect)
with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = [ex.submit(_state_job, j) for j in jobs]
    for done, fut in enumerate(as_completed(futures), 1):
        exp_id, t, res = fut.result()
        if res is None:
            continue
        for var, arr in res.items():
            state[(exp_id, t, var)] = arr
        if done % 24 == 0:
            print(f"  {done}/{len(jobs)}", flush=True)

# CLC comes from section 3, so a state frame needs both reads to have succeeded.
STATE_TIMES = [t for t in FRAME_TIMES
               if all((e, t, v) in state for e in EXP_IDS for v in STATE_READ[:-1] + ["RH"])]
print(f"\n{len(STATE_TIMES)} of {len(FRAME_TIMES)} cycles complete for the state figure")
print(f"held in memory: {sum(a.nbytes for a in state.values()) / 1e6:.0f} MB")

reading 288 cycle-experiment combinations on 16 workers, 1 file each, 4 x 63 messages
  24/288
  48/288
  72/288
  96/288
  120/288
  144/288
  168/288
  192/288
  216/288
  240/288
  264/288
  288/288

144 of 144 cycles complete for the state figure
held in memory: 87 MB


In [12]:
def cycle_inc(exp_id, t, cells=None, levels=None):
    """The assimilation increment on the transect columns: T, QV, QC.

    A cell of its own rather than an addition to the read above, so that gaining the
    increment columns does not mean reading the first-guess files again.
    """
    cells = t_idx if cells is None else cells
    levels = LEVELS_OF_INTEREST if levels is None else levels
    inc = Grib(file_path(exp_id, "inc", t))
    return {v: inc.profile(v, levels)[1][:, cells].astype(np.float32) for v in INC_VARS}


def _inc_job(job):
    exp_id, t = job
    try:
        return exp_id, t, cycle_inc(exp_id, t)
    except Exception as e:
        print(f"  skipped {exp_id} {t:%Y-%m-%d %H}: {type(e).__name__}: {e}", flush=True)
        return exp_id, t, None


print(f"reading {len(jobs)} cycle-experiment combinations on {N_WORKERS} workers, "
      f"1 file each, {len(INC_VARS)} x {len(LEVELS_OF_INTEREST)} messages")

inc_state = {}                              # (exp, time, var) -> (n_levels, n_transect)
with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = [ex.submit(_inc_job, j) for j in jobs]
    for done, fut in enumerate(as_completed(futures), 1):
        exp_id, t, res = fut.result()
        if res is None:
            continue
        for var, arr in res.items():
            inc_state[(exp_id, t, var)] = arr
        if done % 24 == 0:
            print(f"  {done}/{len(jobs)}", flush=True)

# A frame needs every column, so it needs all three reads: CLC, first guess and increment.
FIG_TIMES = [t for t in STATE_TIMES
             if all((e, t, v) in inc_state for e in EXP_IDS for v in INC_VARS)]
print(f"\n{len(FIG_TIMES)} of {len(STATE_TIMES)} cycles complete for every column")
print(f"increments held in memory: {sum(a.nbytes for a in inc_state.values()) / 1e6:.0f} MB")

reading 288 cycle-experiment combinations on 16 workers, 1 file each, 3 x 63 messages
  24/288
  48/288
  72/288
  96/288
  120/288
  144/288
  168/288
  192/288
  216/288
  240/288
  264/288
  288/288

144 of 144 cycles complete for every column
increments held in memory: 65 MB


In [13]:
def state_field(src, var, exp_id, t):
    """One column of the state figure for one experiment.

    Three sources, each already in memory: the increment from the cell above, CLC from the
    section-3 read, everything else from the first-guess read.
    """
    if src == "inc":
        return inc_state[(exp_id, t, var)]
    if var == "CLC":
        return clc[(exp_id, t, "lff")]
    return state[(exp_id, t, var)]


# Colour limits per column, over every frame at once and then frozen, as in section 5.
# First-guess columns are states: the 1st-99th percentile rather than min-max, since a few
# extreme cells would otherwise flatten the contrast everywhere else, and CLC keeps the fixed
# 0-100 % of its own figure. Increment columns are signed corrections, so they get a
# symmetric diverging scale instead - zero has to read as zero.
col_kw, col_diff_kw = {}, {}
for src, var in STATE_COLS:
    if src == "inc":
        v = sym_limit([state_field(src, var, e, t) for e in EXP_IDS for t in FIG_TIMES])
        col_kw[(src, var)] = dict(cmap="RdBu_r", vmin=-v, vmax=v)
        span = f"+-{v:.4g}"
    else:
        vals = np.concatenate([state_field(src, var, e, t).ravel()
                               for e in EXP_IDS for t in FIG_TIMES])
        lo, hi = (0.0, 100.0) if var == "CLC" else np.nanpercentile(vals, [1.0, 99.0])
        col_kw[(src, var)] = dict(cmap=VAR_CMAP[var], vmin=lo, vmax=hi)
        span = f"{lo:.4g} to {hi:.4g}"
    vd = sym_limit([state_field(src, var, EXP_IDS[0], t) - state_field(src, var, EXP_IDS[1], t)
                    for t in FIG_TIMES])
    col_diff_kw[(src, var)] = dict(cmap="PuOr_r", vmin=-vd, vmax=vd)
    print(f"{SRC_LABEL[src]:7s} {var:4s} [{VAR_UNITS[var]:>7s}]  {span:>24s}   "
          f"{EXP_DIFF} +-{vd:.4g}")


def state_section_figure(t):
    """The section figure for one cycle: columns of variables, rows of experiments.

    Transposed against section 5 on purpose - columns of three rows suit a wide screen, where
    stacked rows did not. The first-guess columns come first, the increment columns to their
    right.
    """
    fig = plt.figure(figsize=(34, 13.5))
    gs = fig.add_gridspec(len(CLC_COLS) + 1, len(STATE_COLS),
                          height_ratios=[1] * len(CLC_COLS) + [0.42])
    axes = np.empty((len(CLC_COLS), len(STATE_COLS)), dtype=object)
    for r in range(len(CLC_COLS)):
        for c in range(len(STATE_COLS)):
            ref = axes[0, 0] if (r or c) else None
            axes[r, c] = fig.add_subplot(gs[r, c], sharex=ref, sharey=ref)
            axes[r, c].label_outer()
    ax_bias = fig.add_subplot(gs[len(CLC_COLS), :])

    for c, (src, var) in enumerate(STATE_COLS):
        for r, exp_col in enumerate(CLC_COLS):
            ax = axes[r, c]
            if exp_col == EXP_DIFF:
                field = (state_field(src, var, EXP_IDS[0], t)
                         - state_field(src, var, EXP_IDS[1], t))
                kw = col_diff_kw[(src, var)]
                cb_label = f"{EXP_DIFF} of {SRC_LABEL[src]} {var}"
            else:
                field, kw = state_field(src, var, exp_col, t), col_kw[(src, var)]
                cb_label = f"{SRC_LABEL[src]} {var}"
            pc = ax.pcolormesh(dist2d, z_sec, field, shading="auto", **kw)
            cb = plt.colorbar(pc, ax=ax, shrink=0.88, pad=0.02)
            cb.set_label(f"{cb_label}  [{VAR_UNITS[var]}]", fontsize=7)
            cb.ax.tick_params(labelsize=6.5)
            ax.fill_between(dist_t, 0, terrain, color=TERRAIN_FILL, zorder=5)
            ax.plot(dist_t, terrain, color=TERRAIN_EDGE, lw=0.8, zorder=5)
            ax.axvline(dist_t[i_pay], color="k", ls=":", lw=1.2, zorder=6)
            ax.text(dist_t[i_pay], SECTION_ZTOP * 0.95, " Payerne", fontsize=7,
                    va="top", ha="left", zorder=7)
            ax.set_title(f"{SRC_LABEL[src]} {var}; {exp_col}", fontsize=9)
            ax.set_ylim(0, SECTION_ZTOP)
            if r == len(CLC_COLS) - 1:
                ax.set_xlabel("distance along section [km]", fontsize=9)
            if c == 0:
                ax.set_ylabel(f"{exp_col}\naltitude [m above sea level]", fontsize=9)

    draw_bias_panel(ax_bias, t)
    ax_bias.set_xlabel("valid time [UTC]   |   star = the cycle drawn above")

    plt.suptitle("First guess lff(T) and assimilation increment inc(T) on the west-east "
                 "section through Payerne\n"
                 f"valid {t:%a %d %b %Y  %H} UTC   |   left {len(STATE_ROWS)} columns: the "
                 f"first-guess state, the cloud scheme's inputs through to the cloud fraction "
                 f"it diagnoses; right {len(INC_VARS)} columns: the increment added to it, red "
                 f"adds; bottom row: orange = {EXP_IDS[0]} higher than {EXP_IDS[1]}; "
                 f"grey = terrain   |   section {TRANSECT[0][0]:.2f}N {TRANSECT[0][1]:.2f}E "
                 f"(Lake Geneva) -> {TRANSECT[1][0]:.2f}N {TRANSECT[1][1]:.2f}E (St. Gallen)")
    # rect leaves room for the two-line suptitle, which tight_layout does not reserve
    plt.tight_layout(rect=(0, 0, 1, 0.965))
    return fig


lff(T)  T    [      K]              226.9 to 287   801 - 802 +-1.206
lff(T)  QV   [  kg/kg]      2.84e-05 to 0.008413   801 - 802 +-0.001411
lff(T)  QC   [  kg/kg]            0 to 0.0003369   801 - 802 +-0.0003309
lff(T)  RH   [      %]            5.447 to 100.1   801 - 802 +-26.54
lff(T)  CLC  [      %]                  0 to 100   801 - 802 +-75.57
inc(T)  T    [      K]                  +-0.8709   801 - 802 +-0.8622
inc(T)  QV   [  kg/kg]               +-0.0006459   801 - 802 +-0.0007243
inc(T)  QC   [  kg/kg]               +-0.0002573   801 - 802 +-0.0003287


In [14]:
# Write the state frames, same naming convention as the CLC series but in their own folder.
for n, t in enumerate(FIG_TIMES, 1):
    fig = state_section_figure(t)
    if SAVE_FRAMES:
        fig.savefig(STATE_FRAME_DIR / f"lff_state_{t:%Y%m%d%H}.png", dpi=FRAME_DPI)
    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig)
    if n % 12 == 0 or n == len(FIG_TIMES):
        print(f"  {n}/{len(FIG_TIMES)} frames", flush=True)

state_frames = sorted(STATE_FRAME_DIR.glob("lff_state_*.png"))
sizes = {tuple(plt.imread(f).shape[:2]) for f in state_frames[:5]}
print(f"\n{len(state_frames)} PNGs in {STATE_FRAME_DIR.resolve()}")
print(f"frame size {sizes} (one entry = every frame the same size)")

  12/144 frames
  24/144 frames
  36/144 frames
  48/144 frames
  60/144 frames
  72/144 frames
  84/144 frames
  96/144 frames
  108/144 frames
  120/144 frames
  132/144 frames
  144/144 frames

144 PNGs in /users/jdelbeke/Spin-Up-Down-Project/scripts/figures/lff_state_frames
frame size {(1485, 3740)} (one entry = every frame the same size)
